In [0]:
# DBTITLE 1,Configuração dos Caminhos e Variáveis
from pyspark.sql.functions import current_timestamp

CATALOG = "workspace"
SCHEMA = "default"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data"

datasets = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "payments": "olist_order_payments_dataset.csv"
}

# DBTITLE 2,Ingestão para a Camada Bronze (Delta Lake)
for entity, file_name in datasets.items():
    file_path = f"{VOLUME_PATH}/{file_name}"
    target_table = f"{CATALOG}.{SCHEMA}.bronze_{entity}"
    
    print(f"Lendo {file_name} de {file_path}...")
    
    df_raw = (spark.read
              .option("header", "true")
              .option("inferSchema", "true")
              .csv(file_path))
    
    # Adiciona metadados de auditoria (data/hora de ingestão)
    df_bronze = df_raw.withColumn("_ingested_at", current_timestamp())
    
    # Grava como Tabela Delta na camada Bronze
    (df_bronze.write
     .format("delta")
     .mode("overwrite")
     .saveAsTable(target_table))
    
    print(f"✓ Tabela {target_table} criada com sucesso!\n")

print("--- Ingestão da Camada Bronze Concluída com Sucesso! ---")

Lendo olist_orders_dataset.csv de /Volumes/workspace/default/raw_data/olist_orders_dataset.csv...
✓ Tabela workspace.default.bronze_orders criada com sucesso!

Lendo olist_order_items_dataset.csv de /Volumes/workspace/default/raw_data/olist_order_items_dataset.csv...
✓ Tabela workspace.default.bronze_order_items criada com sucesso!

Lendo olist_products_dataset.csv de /Volumes/workspace/default/raw_data/olist_products_dataset.csv...
✓ Tabela workspace.default.bronze_products criada com sucesso!

Lendo olist_customers_dataset.csv de /Volumes/workspace/default/raw_data/olist_customers_dataset.csv...
✓ Tabela workspace.default.bronze_customers criada com sucesso!

Lendo olist_order_payments_dataset.csv de /Volumes/workspace/default/raw_data/olist_order_payments_dataset.csv...
✓ Tabela workspace.default.bronze_payments criada com sucesso!

--- Ingestão da Camada Bronze Concluída com Sucesso! ---
